In [ ]:
# =========================
# Importaciones
# =========================
import pandas as pd
import numpy as np

RAW = "../../data/raw/"

# =========================
# Carga de datos
# =========================
products = pd.read_csv(RAW + "olist_products_dataset.csv")

print(f"Filas iniciales: {len(products):,}")

# =========================
# Duplicados 
# =========================
products = products.drop_duplicates(subset=["product_id"], keep="first").copy()
print(f"Tras drop_duplicates: {len(products):,}")

# =========================
# Nulos 
# =========================

# Categóricos con nulos: product_category_name
if products["product_category_name"].isna().any():
    products["product_category_name"] = products["product_category_name"].fillna("Unknown")

# Numéricos con nulos cambiado por la mediana
num_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]

for c in num_cols:
    if products[c].isna().any():
        med = products[c].median()
        products[c] = products[c].fillna(med)


# =========================
# 3) Outliers ±3σ 
# =========================
def remove_outliers_3sigma(df, cols):
    if not cols:
        return df
    mask = pd.Series(True, index=df.index)
    for c in cols:
        x = df[c].astype(float)
        mu = x.mean()
        sigma = x.std(ddof=0)
        # si sigma=0 o NaN, no filtramos por esa columna
        if np.isnan(mu) or np.isnan(sigma) or sigma == 0:
            continue
        mask &= (x >= mu - 3*sigma) & (x <= mu + 3*sigma)
    return df[mask].copy()

before = len(products)
products_clean = remove_outliers_3sigma(products, num_cols)
print(f"Eliminadas por outliers (±3σ) en {num_cols}: {before - len(products_clean):,}")

# =========================
# 4) Resultado final
# =========================
print(f"Filas finales: {len(products_clean):,}")
print(products_clean.dtypes)
products_clean.head()
